In [1]:
import pandas as pd
import numpy as np 
from haversine import haversine, Unit
import math

In [2]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from scripts.PHMM.viterbi_algorithm.states import *
from scripts.PHMM.viterbi_algorithm.viterbi import *
from scripts.PHMM.distributions.AIS_RF_distribution import *
import pandas as pd

In [4]:
# segmented tracks 
segmented_tracks = pd.read_parquet("../data/processed/segmented_tracks.parquet")
segmented_tracks.head(1)

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass,TimeDiff,SubTrackID
18149901,209112000,2024-01-03 13:19:58,48.51548,-126.51201,9.8,98.7,104.0,STALO,IMO9758404,5BQM4,70.0,0.0,229.0,32.0,7.5,70.0,A,NaT,0


In [5]:
results = []

for (mmsi, subtrack_id), track_data in segmented_tracks.groupby(["MMSI", "SubTrackID"]):

    track_data["prev_LAT"] = track_data["LAT"].shift(1)
    track_data["prev_LON"] = track_data["LON"].shift(1)
    track_data["distance_km"] = track_data.apply(lambda row:
        haversine(
            (row["prev_LAT"], row["prev_LON"]),
            (row["LAT"], row["LON"])
    ), axis= 1)

    track_data["time_sec"] = track_data["TimeDiff"].dt.total_seconds()

    track_data["speed_kmph"] = np.where(
        (track_data["time_sec"] > 0) & (track_data["distance_km"] > 0),
        (track_data["distance_km"]/track_data["time_sec"]) * 3600,
        0.0
    )
    
    results.append(track_data)
    
df_results = pd.concat(results)
df_results.head(5)

,MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,...,Draft,Cargo,TransceiverClass,TimeDiff,SubTrackID,prev_LAT,prev_LON,distance_km,time_sec,speed_kmph
18149901,209112000,2024-01-03 13:19:58,48.51548,-126.51201,9.8,98.7,104.0,STALO,IMO9758404,5BQM4,...,7.5,70.0,A,NaT,0,NaN,NaN,NaN,NaN,0.000000
18193127,209112000,2024-01-03 13:32:50,48.51033,-126.45776,10.3,99.7,104.0,STALO,IMO9758404,5BQM4,...,7.5,70.0,A,0 days 00:12:52,0,48.51548,-126.51201,4.036950,772.0,18.825155
18180531,209112000,2024-01-03 13:34:50,48.50968,-126.44914,10.5,91.4,104.0,STALO,IMO9758404,5BQM4,...,7.5,70.0,A,0 days 00:02:00,0,48.51033,-126.45776,0.639097,120.0,19.172914
18199859,209112000,2024-01-03 13:40:59,48.50753,-126.42323,10.1,96.0,107.0,STALO,IMO9758404,5BQM4,...,7.5,70.0,A,0 days 00:06:09,0,48.50968,-126.44914,1.923641,369.0,18.767225
18231921,209112000,2024-01-03 13:42:38,48.50664,-126.41652,9.8,107.4,107.0,STALO,IMO9758404,5BQM4,...,7.5,70.0,A,0 days 00:01:39,0,48.50753,-126.42323,0.504133,99.0,18.332117


In [6]:
# removed blocks, dus PHMM sequence data
PHMM_sequence = pd.read_parquet("../data/processed/PHMM_sequence_data.parquet")
PHMM_sequence.head(1)

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
0,0,209112000,0,2024-01-03 13:19:57,NaT,NaT,None,None,begin


In [7]:
results2 = []

for track_id, track_data in PHMM_sequence.groupby("TrackID"):
    
    AIS_sequence = [(time, coord) 
                    for time, coord in zip(track_data["AIS_Timestamp"], track_data["AIS"]) 
                    if pd.notna(time) and coord is not None]

    for i in range(len(AIS_sequence)-1):
        time_curr_row, coord_curr_row = AIS_sequence[i]
        time_next_row, coord_next_row = AIS_sequence[i+1]
        
        delta_time_sec = (time_next_row - time_curr_row).total_seconds()
        delta_dist_km = haversine(coord_curr_row, coord_next_row)
        
        if delta_time_sec > 0 and delta_dist_km > 0:
            speed_kmph = delta_dist_km / (delta_time_sec / 3600)
            results2.append({
                "TrackID": track_id,
                "curr_time": time_curr_row,
                "next_time": time_next_row,
                "curr_coord": coord_curr_row,
                "next_coord": coord_next_row,
                "delta_time": delta_time_sec,
                "delta_dist": delta_dist_km,
                "speed_kmph": speed_kmph
            }) 

df_results2 = pd.DataFrame(results2)
df_results2.head(5)

,TrackID,curr_time,next_time,curr_coord,next_coord,delta_time,delta_dist,speed_kmph
0,0,2024-01-03 13:19:58,2024-01-03 13:32:50,"[48.51548, -126.51201]","[48.51033, -126.45776]",772.0,4.036950,18.825155
1,0,2024-01-03 13:32:50,2024-01-03 13:34:50,"[48.51033, -126.45776]","[48.50968, -126.44914]",120.0,0.639097,19.172914
2,0,2024-01-03 13:34:50,2024-01-03 13:40:59,"[48.50968, -126.44914]","[48.50753, -126.42323]",369.0,1.923641,18.767225
3,0,2024-01-03 13:40:59,2024-01-03 13:42:38,"[48.50753, -126.42323]","[48.50664, -126.41652]",99.0,0.504133,18.332117
4,0,2024-01-03 13:42:38,2024-01-03 13:43:50,"[48.50664, -126.41652]","[48.50607, -126.41174]",72.0,0.357805,17.890253


In [8]:
begin = BeginState()
AIS = AISState()
RF = RFState()
M = MState()
end = EndState()

states = [begin, AIS, RF, M, end]
model = PHMM(states, end)

In [9]:
first_200_trackids = PHMM_sequence["TrackID"].drop_duplicates().head(200)
df_first_200 = PHMM_sequence[PHMM_sequence["TrackID"].isin(first_200_trackids)]

In [10]:
RF_data = df_first_200[df_first_200["RF"].notna()].copy()
RF_data["RF_data_point_id"] = RF_data.groupby("TrackID").cumcount() + 1

M_data = df_first_200[df_first_200["State"] == "M"].copy()
M_data["distance_km"] = M_data.apply(lambda row: haversine(row["AIS"], row["RF"], unit=Unit.KILOMETERS), axis=1)
radius_km = math.ceil(M_data["distance_km"].max())

viterbi_results = []

for _, RF_data_point in RF_data.iterrows():
    RF_coord = RF_data_point["RF"]
    RF_time = RF_data_point["RF_Timestamp"]
    
    for AIS_id, AIS_data in df_first_200.groupby("TrackID"):
        AIS_coords = [coord for coord in AIS_data["AIS"] if coord is not None]
        RF_within_radius = any(haversine(RF_coord, coord) <= radius_km for coord in AIS_coords)
        
        marge = pd.Timedelta(minutes=15)
        min_AIS_time = AIS_data["AIS_Timestamp"].min() - marge
        max_AIS_time = AIS_data["AIS_Timestamp"].max() + marge
        RF_within_time = min_AIS_time <= RF_time <= max_AIS_time
        
        if RF_within_radius and RF_within_time:
            AIS_seq = [(time, coord) 
                       for time, coord in zip(AIS_data["AIS_Timestamp"], AIS_data["AIS"]) 
                       if pd.notna(time) and coord is not None]
            RF_seq = [(RF_time, RF_coord)]
            
            path, prob, log_probs = model.viterbi(AIS_seq, RF_seq)
            normalized_prob = prob/len(AIS_seq)
            
            viterbi_results.append(
                {
                    "RF_sequence_id": RF_data_point["TrackID"],
                    "RF_data_point_id": RF_data_point["RF_data_point_id"],
                    "AIS_sequence_id": AIS_id,
                    "num_of_AIS_points": len(AIS_seq),
                    "viterbi_prob": prob,
                    "normalized_viterbi_prob": normalized_prob,
                    "match": RF_data_point["TrackID"] == AIS_id,
                    "predicted_path": path
                }) 

[RESET] gap_counter = 0
AIS_to_M: 1
AIS_to_M: 2
RF_to_AIS: 3
AIS_to_M: 4
AIS_to_M: 5
AIS_to_M: 6
RF_to_AIS: 7
AIS_to_M: 8
AIS_to_M: 9
AIS_to_M: 10
RF_to_AIS: 11
AIS_to_M: 12
AIS_to_M: 13
AIS_to_M: 14
RF_to_AIS: 15
AIS_to_M: 16
AIS_to_M: 17
AIS_to_M: 18
RF_to_AIS: 19
AIS_to_M: 20
AIS_to_M: 21
AIS_to_M: 22
RF_to_AIS: 23
AIS_to_M: 24
AIS_to_M: 25
AIS_to_M: 26
RF_to_AIS: 27
AIS_to_M: 28
AIS_to_M: 29
AIS_to_M: 30
RF_to_AIS: 31
AIS_to_M: 32
AIS_to_M: 33
AIS_to_M: 34
RF_to_AIS: 35
AIS_to_M: 36
AIS_to_M: 37
AIS_to_M: 38
RF_to_AIS: 39
AIS_to_M: 40
AIS_to_M: 41
AIS_to_M: 42
RF_to_AIS: 43
AIS_to_M: 44
AIS_to_M: 45
AIS_to_M: 46
RF_to_AIS: 47
AIS_to_M: 48
AIS_to_M: 49
AIS_to_M: 50
RF_to_AIS: 51
AIS_to_M: 52
AIS_to_M: 53
AIS_to_M: 54
RF_to_AIS: 55
AIS_to_M: 56
AIS_to_M: 57
AIS_to_M: 58
RF_to_AIS: 59
AIS_to_M: 60
AIS_to_M: 61
AIS_to_M: 62
RF_to_AIS: 63
AIS_to_M: 64
AIS_to_M: 65
AIS_to_M: 66
RF_to_AIS: 67
AIS_to_M: 68
AIS_to_M: 69
AIS_to_M: 70
RF_to_AIS: 71
AIS_to_M: 72
AIS_to_M: 73
AIS_to_M: 74
RF_to

KeyboardInterrupt: 

In [ ]:
for result in viterbi_results:
    track_id = result["AIS_sequence_id"]
    speeds = df_results2[df_results2["TrackID"] == track_id]["speed_kmph"]
    avg_speed = speeds.mean()
    
    result["avg_speed"] = avg_speed
    result["normalized_prob_speed"] = result["normalized_viterbi_prob"]/avg_speed

In [ ]:
df_viterbi_results = pd.DataFrame(viterbi_results)

In [ ]:
ids = df_viterbi_results.groupby(["RF_sequence_id", "RF_data_point_id"])["normalized_viterbi_prob"].idxmax()
best_matches = df_viterbi_results.loc[ids].reset_index(drop=True)
best_matches

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,viterbi_prob,normalized_viterbi_prob,match,predicted_path,avg_speed,normalized_prob_speed
0,0,1,0,885,-29.816853,-0.033691,True,"[begin, AIS, AIS, AIS, AIS, RF, AIS, AIS, AIS,...",18.102813,-0.001861
1,0,2,0,885,-33.947161,-0.038358,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",18.102813,-0.002119
2,0,3,0,885,-41.404324,-0.046785,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",18.102813,-0.002584
3,1,1,1,883,-35.737613,-0.040473,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",18.957101,-0.002135
4,1,2,1,883,-32.603663,-0.036924,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",18.957101,-0.001948
...,...,...,...,...,...,...,...,...,...,...
490,198,4,119,823,-30.965156,-0.037625,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",10.104258,-0.003724
491,199,1,199,981,-34.709551,-0.035382,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",14.217935,-0.002489
492,199,2,199,981,-39.022244,-0.039778,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",14.217935,-0.002798
493,199,3,199,981,-30.312720,-0.030900,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",14.217935,-0.002173


In [ ]:
best_matches[best_matches["match"] == False].head(5)

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,viterbi_prob,normalized_viterbi_prob,match,predicted_path,avg_speed,normalized_prob_speed
11,4,1,3,646,-47.584147,-0.073660,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",14.580310,-0.005052
13,5,2,79,659,-30.445415,-0.046199,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",27.242030,-0.001696
50,18,3,19,668,-21.226305,-0.031776,False,"[begin, RF, AIS, AIS, AIS, AIS, AIS, AIS, AIS,...",25.034476,-0.001269
54,20,3,21,866,-28.917532,-0.033392,False,"[begin, AIS, AIS, RF, AIS, AIS, AIS, AIS, AIS,...",12.036848,-0.002774
68,26,1,98,787,-87.530806,-0.111221,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",15.194272,-0.007320


In [ ]:
df_viterbi_results[df_viterbi_results["RF_sequence_id"] == 5]

,RF_sequence_id,RF_data_point_id,AIS_sequence_id,num_of_AIS_points,viterbi_prob,normalized_viterbi_prob,match,predicted_path,avg_speed,normalized_prob_speed
12,5,1,5,542,-26.970522,-0.049761,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",15.285883,-0.003255
13,5,2,5,542,-25.471907,-0.046996,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",15.285883,-0.003074
14,5,2,79,659,-30.445415,-0.046199,False,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",27.242030,-0.001696
15,5,3,5,542,-25.034528,-0.046189,True,"[begin, AIS, AIS, AIS, AIS, AIS, AIS, AIS, AIS...",15.285883,-0.003022


In [11]:
PHMM_sequence[PHMM_sequence["TrackID"] == 0]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State
0,0,209112000,0,2024-01-03 13:19:57,NaT,NaT,None,None,begin
1,0,209112000,0,2024-01-03 13:19:58,2024-01-03 13:19:58,NaT,"[48.51548, -126.51201]",None,AIS
2,0,209112000,0,2024-01-03 13:32:50,2024-01-03 13:32:50,NaT,"[48.51033, -126.45776]",None,AIS
3,0,209112000,0,2024-01-03 13:34:50,2024-01-03 13:34:50,NaT,"[48.50968, -126.44914]",None,AIS
4,0,209112000,0,2024-01-03 13:40:59,2024-01-03 13:40:59,2024-01-03 13:41:00,"[48.50753, -126.42323]","[48.499443942360536, -126.4146031322306]",M
...,...,...,...,...,...,...,...,...,...
883,0,209112000,0,2024-01-04 13:03:22,2024-01-04 13:03:22,NaT,"[47.28273, -122.44032]",None,AIS
884,0,209112000,0,2024-01-04 13:12:27,2024-01-04 13:12:27,NaT,"[47.28264, -122.44023]",None,AIS
885,0,209112000,0,2024-01-04 13:15:26,2024-01-04 13:15:26,NaT,"[47.28258, -122.44013]",None,AIS
886,0,209112000,0,2024-01-04 13:18:25,2024-01-04 13:18:25,NaT,"[47.28257, -122.44008]",None,AIS


In [ ]:
RF_data[RF_data["TrackID"] == 5]

,TrackID,MMSI,SubTrackID,Timestamp,AIS_Timestamp,RF_Timestamp,AIS,RF,State,RF_data_point_id
3493,5,210614000,2,2024-01-03 01:41:00,NaT,2024-01-03 01:41:00,None,"[36.96347091030869, -76.4119991966966]",RF,1
3679,5,210614000,2,2024-01-03 09:41:37,2024-01-03 09:41:37,2024-01-03 09:41:00,"[36.91305, -75.91711]","[36.917909658337884, -75.90767513607126]",M,2
3809,5,210614000,2,2024-01-03 13:41:47,2024-01-03 13:41:47,2024-01-03 13:41:00,"[36.61866, -75.19177]","[36.61523605648769, -75.18262300003032]",M,3
4003,5,210614000,2,2024-01-03 21:41:00,NaT,2024-01-03 21:41:00,None,"[35.98732355336836, -73.69037119456289]",RF,4
